# The Analysis
Read the analyzed data from Google Sheets, calculate the insights, and generate a plain-language summary.

In [7]:
import gspread, yaml, json
import pandas as pd
from openai import OpenAI
from pprint import pprint

In [8]:
pd.set_option('display.max_rows', None)

In [9]:
# open a gspread connection using the service account credentials file
gc = gspread.service_account(filename='service-account-credentials.json')

In [10]:
# open the Google Sheet by its URL
spreadsheet = gc.open_by_url('https://docs.google.com/spreadsheets/d/1ivp6wQ25X9F21BFA38ieNqN_BVHpBFbGThxduzmZHLI/edit?usp=sharing')

In [11]:
# get the worksheet object that represents the first worksheet/tab (index 0) in the Google Sheet Spreasheet sh
worksheet = spreadsheet.get_worksheet(0)

In [12]:
list_of_dicts = worksheet.get_all_records()

# Pandas Analysis
## Phase 2 — Summary Layer

1. Read categorized data from Google Sheet
2. Identify top recurring issues across all reviews — frequency count of each `issue_type`
3. Calculate week-over-week frequency changes — group by week, count each `issue_type` per week, compare to previous week
4. Generate plain-language summary text — pass the calculated numbers to the LLM to write a readable summary for a non-technical reader

In [13]:
df = pd.DataFrame(list_of_dicts)
df.head()


,date,issue_type,severity,note
0,2026-01-22,food_quality,high,The pizza and shawarma were described as extre...
1,2025-11-13,positive,none,"Customer praised the food, saying the Chicken ..."
2,2024-01-24,food_quality,low,"The pizza tasted good, but the bread base was ..."
3,2024-06-28,positive,none,Customer praised the pizza quality (crispy cru...
4,2026-02-02,positive,none,Customer praised the friendly service and deli...


In [14]:
print(df.shape,'\n')        # How many rows and columns?    # Show first 5 rows
print(df.info(),'\n')            # Column names, data types, null count
print(df.isnull().sum())    # Spot any missing values in each column

(127, 4) 

<class 'pandas.DataFrame'>
RangeIndex: 127 entries, 0 to 126
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        127 non-null    str  
 1   issue_type  127 non-null    str  
 2   severity    127 non-null    str  
 3   note        127 non-null    str  
dtypes: str(4)
memory usage: 4.1 KB
None 

date          0
issue_type    0
severity      0
note          0
dtype: int64


In [15]:
df['issue_type'].value_counts(ascending=False) # how many times each issue type appears in the dataset?

issue_type
positive        80
food_quality    18
pricing         11
service         10
ambiance         6
cleanliness      2
Name: count, dtype: int64

In [33]:
# convert the `date` column to datetime objects for time-based analysis
df['date'] = pd.to_datetime(df['date'])

In [17]:
# Converts each date row-by-row into the week it belongs to (e.g. 2026-05-11/2026-05-17)
weeks = df['date'].dt.to_period('W')
weeks.head()

0    2026-01-19/2026-01-25
1    2025-11-10/2025-11-16
2    2024-01-22/2024-01-28
3    2024-06-24/2024-06-30
4    2026-02-02/2026-02-08
Name: date, dtype: period[W-SUN]

In [18]:
# For each week-group, counts how many times each issue type appears going down the rows
ft_issueType_byWeek = (df.groupby(weeks)['issue_type']
    .value_counts()
    .reset_index(name='count')# Sorts row-by-row: first by most recent week, then by highest count within each week
    .sort_values(['count', 'date'], ascending=[False, False])
)
ft_issueType_byWeek

,date,issue_type,count
13,2024-07-29/2024-08-04,positive,4
91,2026-03-02/2026-03-08,positive,3
42,2025-03-24/2025-03-30,positive,3
99,2026-04-27/2026-05-03,positive,2
95,2026-04-06/2026-04-12,positive,2
88,2026-02-02/2026-02-08,positive,2
81,2026-01-05/2026-01-11,positive,2
65,2025-09-22/2025-09-28,positive,2
62,2025-09-15/2025-09-21,positive,2
61,2025-09-08/2025-09-14,pricing,2


Calculate week-over-week frequency changes — group by week, count each `issue_type` per week, compare to previous week

In [19]:
ft_issueType_sorted = ft_issueType_byWeek.sort_values(['issue_type', 'date'], ascending=[True, True]).reset_index(drop=True)
ft_issueType_sorted

,date,issue_type,count
0,2025-01-06/2025-01-12,ambiance,1
1,2025-02-24/2025-03-02,ambiance,1
2,2025-03-31/2025-04-06,ambiance,1
3,2025-07-07/2025-07-13,ambiance,1
4,2025-10-27/2025-11-02,ambiance,1
5,2026-02-23/2026-03-01,ambiance,1
6,2024-05-27/2024-06-02,cleanliness,1
7,2025-11-10/2025-11-16,cleanliness,1
8,2024-01-22/2024-01-28,food_quality,1
9,2024-06-10/2024-06-16,food_quality,1


In [20]:
# create a Series that calculates the percentage change in count for each issue type compared to the previous week
pct_change = ft_issueType_sorted.groupby('issue_type')['count'].pct_change(fill_method=None) * 100

In [21]:
# add the calculated percentage change as a new column in the DataFrame
ft_issueType_sorted['pct_change'] = pct_change.round(2)
ft_issueType_sorted.head(10)

,date,issue_type,count,pct_change
0,2025-01-06/2025-01-12,ambiance,1,NaN
1,2025-02-24/2025-03-02,ambiance,1,0.0
2,2025-03-31/2025-04-06,ambiance,1,0.0
3,2025-07-07/2025-07-13,ambiance,1,0.0
4,2025-10-27/2025-11-02,ambiance,1,0.0
5,2026-02-23/2026-03-01,ambiance,1,0.0
6,2024-05-27/2024-06-02,cleanliness,1,NaN
7,2025-11-10/2025-11-16,cleanliness,1,0.0
8,2024-01-22/2024-01-28,food_quality,1,NaN
9,2024-06-10/2024-06-16,food_quality,1,0.0


In [22]:
# Calculate the total number of issues for each week by summing the 'count' column grouped by 'date'
total_issues = ft_issueType_sorted.groupby('date')['count'].transform('sum')
# Add the total number of issues as a new column in the DataFrame
ft_issueType_sorted['total_issues'] = total_issues

# Calculate the percentage of each issue type relative to the total issues for that week
ft_issueType_sorted['pct_of_weekly_total'] = (ft_issueType_sorted['count'] / ft_issueType_sorted['total_issues'] * 100).round(2)
ft_issueType_sorted.head()

,date,issue_type,count,pct_change,total_issues,pct_of_weekly_total
0,2025-01-06/2025-01-12,ambiance,1,NaN,1,100.00
1,2025-02-24/2025-03-02,ambiance,1,0.0,2,50.00
2,2025-03-31/2025-04-06,ambiance,1,0.0,2,50.00
3,2025-07-07/2025-07-13,ambiance,1,0.0,3,33.33
4,2025-10-27/2025-11-02,ambiance,1,0.0,1,100.00


In [23]:
# Export the final DataFrame to a CSV string without the index
final_report_csv = ft_issueType_sorted.to_csv(None, index=False)

In [24]:
system_role = f"""
You are a senior restaurant operations consultant with over 15 years of experience advising independent and chain restaurants across the Middle East and Europe. You specialize in translating customer feedback data into clear, commercially relevant insights that restaurant owners can act on.

Your writing style is direct, professional, and grounded — like a trusted advisor speaking to a client, not a data analyst presenting a spreadsheet. You do not use bullet points or numbered lists. You write in clear paragraphs. You avoid jargon. You speak plainly about what the numbers mean for the business.

When interpreting data, you understand that operational issues in restaurants are rarely isolated — slow service often correlates with kitchen throughput problems, food quality complaints often spike during high-volume periods, and pricing friction tends to intensify when food quality also declines. You use this operational knowledge to explain the "why" behind the numbers, not just the "what."

Your report must:
- Open with a concise executive summary (2–3 sentences) that gives the owner the big picture immediately.
- Analyze trends over time using the weekly data provided, referencing specific percentage changes and relative frequencies where relevant.
- Interpret what the numbers likely mean operationally — do not just restate the data.
- Close with exactly one clear, prioritized next step. This must be a specific, actionable first action — not a list of recommendations.

Do not fabricate data. Do not add information not present in the CSV. Do not reference review platforms or sources.
"""

user_role = f"""
Below is a weekly summary of customer complaint data for La Deliziosa Pizzeria, a mid-range Italian restaurant. The data has been extracted and structured from customer reviews.
Each row represents the number of times a specific issue type was mentioned in a given week, along with how that count changed compared to the previous week and how much that issue contributed to all complaints that week.

CSV data:

{final_report_csv}

Column definitions:
- date: the week starting date (YYYY-MM-DD)
- issue_type: the category of the operational issue raised by customers
- count: number of times this issue was mentioned that week
- pct_change: percentage change in count compared to the previous week (positive = increase, negative = decrease)
- total_issues: total number of issue mentions across all categories for that week
- pct_of_weekly_total: this issue type's share of all complaints that week, as a percentage

Write a report for the restaurant owner. The report should read as if it was written by a senior industry consultant who has reviewed this data and is presenting findings in a client briefing. Use a professional but human tone — not robotic, not academic. Write in paragraphs only. No bullet points. No numbered lists.

The report must cover:
1. An executive summary: what is happening overall, in plain language.
2. Trend analysis: which issues are growing, which are stable, which are declining — and what that pattern likely signals operationally.
3. Context and interpretation: explain the likely reasons behind the numbers, drawing on how restaurants typically operate.
4. One priority action: the single most important thing the owner should do first, stated clearly and specifically.

Keep the tone of a consultant who respects the owner's time and gets to the point.
"""

In [25]:
# read config data from config.yaml file
with open('config.yaml', 'r') as file:
    config_data = yaml.safe_load(file)
    api_key = config_data['api_keys']['openai']

# instantiate the OpenAI client object with the API key from the config file
client = OpenAI(api_key=api_key)

response = client.chat.completions.create(
    model= "gpt-5.4-nano-2026-03-17",
   
    messages = [
        # first obect is the "system" role to proovide a persona and instructions to the LLM before the conversation starts. 
        {
            "role": "system",
            "content": system_role
        },
        # second object is the "user" role to provide the acutal input, in this case the review to analyze.
        {
            "role": "user",
            "content": user_role
        }
    ]
)

In [26]:
pprint(json.dumps(response.model_dump(), indent=2))

('{\n'
 '  "id": "chatcmpl-DhX9V55Hhmv38Q1ltKw8YjfnnnwlE",\n'
 '  "choices": [\n'
 '    {\n'
 '      "finish_reason": "stop",\n'
 '      "index": 0,\n'
 '      "logprobs": null,\n'
 '      "message": {\n'
 '        "content": "La Deliziosa\\u2019s complaint picture is not dominated '
 'by one single runaway problem. Across the weeks where complaints are '
 'recorded, the biggest operational signal is food quality, which repeatedly '
 'makes up the largest share of total complaints in several low- and '
 'high-volume weeks. Service and pricing complaints show intermittent spikes, '
 'while ambiance and cleanliness appear far less frequently.\\n\\nLooking at '
 'what\\u2019s changing over time, food quality is the clear pattern-holder. '
 'It shows multiple weeks at higher intensity than the surrounding weeks, most '
 'notably when it reaches 2 mentions in the week of 2025-06-23/2025-06-29 '
 '(66.67% of that week\\u2019s complaints), and it also jumps up materially in '
 'the week of 20

In [27]:
full_report = response.choices[0].message.content.strip()


## Writing Full Report to Google Docs

In [28]:
from googleapiclient.discovery import build
from google.oauth2.service_account import Credentials

In [29]:
# authentication code
SCOPES = ['https://www.googleapis.com/auth/documents']
creds = Credentials.from_service_account_file('service-account-credentials.json', scopes=SCOPES)
docs_service = build('docs', 'v1', credentials=creds)

In [30]:
DOCUMENT_ID = "16vESKsWOkUNI6cjzlG8f2NJ_kHQ8U5kv3V0O-YJxNps"

In [31]:
docs_service.documents().batchUpdate(
    documentId=DOCUMENT_ID,
    body={
        'requests': [
            {
                'insertText': {
                    'location': {'index': 1}, # index: 1 means insert the text at position 1, which is the very first character position in the document body
                    'text': full_report
                }
            }
        ]
    }
).execute()

{'replies': [{}],
 'writeControl': {'requiredRevisionId': 'AFwiY18yHATlN9NxiSAmWLd3fQjzFEyf6SLbFfpBNsLFegYWAjomj-InOJBryURRPuvuAKdxZA1RrdpaZ439vse8HiaCvYrwjAlyPbCzw7hg80I'},
 'documentId': '16vESKsWOkUNI6cjzlG8f2NJ_kHQ8U5kv3V0O-YJxNps'}